In [ ]:
import sys

sys.path.append("..")
sys.path.insert(0, "../../wildlife-datasets")
sys.path.insert(0, "../../wildlife-tools")

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from animal_re_id.datasets import TurtlewatchEgypt_New

warnings.filterwarnings("ignore")

In [ ]:
file_name = "../notebooks/individuals.csv"
root = "/data/wildlife_datasets/turtle_watch/citizen"
root_heads = "/data/wildlife_datasets/turtle_watch/PROFILES"
root_flippers_f = "/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR_front"
root_flippers_r = "/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR_rear"
root_new = "/data/wildlife_datasets/turtle_watch"
roots_new = [
    "2017 PICS + DATASHEET",
    "2018 PICS + DATASHEET",
    "2019 PICS + DATASHEET",
    "2020 PICS + DATASHEET",
    "2021 PICS + DATASHEET",
    "2022 PICS + online database",
    "2023 PICS + online database",
    "2024 PICS + online database",
    "2025 PICS + online database",
]

In [ ]:
from animal_re_id.datasets import combine_datasets_twe

dataset_true = combine_datasets_twe(root_new, roots_new, file_name, load_segmentation=False)
mask1 = dataset_true.metadata["identity"] != "unknown"
mask2 = ~dataset_true.metadata["identity"].isnull()
dataset_true = dataset_true.get_subset(mask1 & mask2)
dataset_true.metadata["year"] = dataset_true.metadata["path"].str[:4].astype(int)

encounters_id = dataset_true.metadata["encounter_id"].unique().tolist()
summary = {encounter_id: {} for encounter_id in encounters_id}
for encounter_id, df_encounter in dataset_true.metadata.groupby("encounter_id"):
    identity = df_encounter["identity"].unique().tolist()
    year = df_encounter["year"].unique().tolist()
    if len(identity) != 1:
        raise ValueError("Multiple identities for encounter")
    if len(year) != 1:
        raise ValueError("Multiple years for encounter")
    summary[encounter_id]["identity"] = identity[0]
    summary[encounter_id]["year"] = year[0]

In [ ]:
dataset = TurtlewatchEgypt_New(root_new, roots_new, True, file_name)
metadata = dataset.dataset.metadata
assert metadata["segmentation"].isnull().sum() == 0

In [ ]:
for i in range(5):
    mask1 = metadata["label"] == "head"
    mask2 = metadata["score"] >= 0.8
    mask = np.where(mask1 & mask2)[0]
    np.random.shuffle(mask)
    dataset.dataset.plot_grid(idx=mask, rotate=False)
    plt.savefig(f"heads_{i}.png", dpi=600, bbox_inches="tight")

    mask1 = metadata["label"].isin(["flipper_fl", "flipper_fr"])
    mask2 = metadata["score"] >= 0.8
    mask = np.where(mask1 & mask2)[0]
    np.random.shuffle(mask)
    dataset.dataset.plot_grid(idx=mask, rotate=True)
    plt.savefig(f"front_flippers_{i}.png", dpi=600, bbox_inches="tight")

    mask1 = metadata["label"].isin(["flipper_rl", "flipper_rr"])
    mask2 = metadata["score"] >= 0.8
    mask = np.where(mask1 & mask2)[0]
    np.random.shuffle(mask)
    dataset.dataset.plot_grid(idx=mask, rotate=True)
    plt.savefig(f"rear_flippers_{i}.png", dpi=600, bbox_inches="tight")

In [ ]:
for encounter_id, df_encounter in metadata.groupby("encounter_id"):
    if encounter_id in encounters_id:
        n_heads = (df_encounter["label"] == "head").sum()
        n_front_flippers = (df_encounter["label"].isin(("flipper_fl", "flipper_fr"))).sum()
        n_rear_flippers = (df_encounter["label"].isin(("flipper_rl", "flipper_rr"))).sum()

        summary[encounter_id]["heads"] = n_heads
        summary[encounter_id]["front flippers"] = n_front_flippers
        summary[encounter_id]["rear flippers"] = n_rear_flippers
        summary[encounter_id]["total"] = n_heads + n_front_flippers + n_rear_flippers

df = pd.DataFrame(summary).T
df[df.isnull()] = 0

In [ ]:
years = np.sort(df["year"].unique())[1:]
counts_all = {}
counts_known = {}
for year in years:
    database = df[df["year"] < year]
    query = df[df["year"] == year]

    counts_all[year] = len(query)
    counts_known[year] = query["identity"].isin(database["identity"]).sum()

In [ ]:
plot_df = pd.DataFrame(
    {
        "all": counts_all,
        "known": counts_known,
    }
)
plot_df["known_ratio"] = np.round(100 * plot_df["known"] / plot_df["all"], 1)
plot_df["new"] = plot_df["all"] - plot_df["known"]
display(plot_df)

cols = ["all", "known", "new"]
plot_df[cols].plot(kind="bar")
plt.grid()
plt.gca().set_axisbelow(True)
plt.title("Number of identified encounters per year")
plt.savefig("counts.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def bin_values(s):
    return pd.cut(s, bins=[-1, 0, 1, 2, 3, 4, np.inf], labels=["0", "1", "2", "3", "4", "5+"])


binned = {}
for c in ["heads", "front flippers", "rear flippers", "total"]:
    binned[c] = bin_values(df[c]).value_counts().sort_index()

plot_df = pd.DataFrame(binned)
plot_df.plot(kind="bar")
plt.grid()
plt.gca().set_axisbelow(True)
plt.title("Number of segmented body parts per encounter")
plt.xticks(rotation=0)
plt.savefig("segmentation.png", dpi=300, bbox_inches="tight")
plt.show()